In [1]:
import os
import numpy as np
import pandas as pd
import sys
import time
from datetime import datetime
from pyswarm import pso  # Install with: pip install pyswarm
import warnings
warnings.filterwarnings("ignore")
from utilities.utils import HelperFunctions, SSPModelForCalibration, ErrorFunctions
from utilities.diff_reports_v2 import DiffReportUtils
import logging
from sisepuede.manager.sisepuede_examples import SISEPUEDEExamples
from ssp_transformations_handler.GeneralUtils import GeneralUtils
import json


# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Record the start time
start_time = time.time()

# Initialize helper functions
helper_functions = HelperFunctions()
gu = GeneralUtils()

In [4]:
# Paths
SRC_FILE_PATH = os.getcwd()
build_path = lambda PATH: os.path.abspath(os.path.join(*PATH))
DATA_PATH = build_path([SRC_FILE_PATH, "..", "data"])
OUTPUT_PATH = build_path([SRC_FILE_PATH, "..", "output"])
MISC_FILES_PATH = build_path([SRC_FILE_PATH, 'misc'])
VAR_MAPPING_FILES_PATH = build_path([MISC_FILES_PATH, 'var_mapping'])
SECTORAL_REPORT_PATH = build_path([MISC_FILES_PATH, 'sectoral_reports'])
SECTORAL_REPORT_MAPPING_PATH = build_path([MISC_FILES_PATH, 'sectoral_report_mapping'])
OPT_CONFIG_FILES_PATH = build_path([SRC_FILE_PATH, 'config'])
OPT_OUTPUT_PATH = build_path([SRC_FILE_PATH,"..", "output"])

In [5]:
# Make sure the output directory exists
os.makedirs(OPT_OUTPUT_PATH, exist_ok=True)

In [6]:
# Get important params from the YAML file
try:
    yaml_file = 'uganda_opt_config.yaml'
except IndexError:
    raise ValueError("YAML configuration file must be provided as a command-line argument.")

param_dict = helper_functions.get_parameters_from_yaml(build_path([OPT_CONFIG_FILES_PATH, yaml_file]))

target_region = param_dict['target_region']
iso_alpha_3 = param_dict['iso_alpha_3']
stressed_variables_report_version = param_dict['stressed_variables_report_version']
input_data_file_to_calibrate = param_dict["input_data_file_to_calibrate"]
detailed_diff_report_flag = param_dict['detailed_diff_report_flag']
energy_model_flag = param_dict['energy_model_flag']
use_edgar_db_flag = param_dict['use_edgar_db_flag']
sim_init_year = param_dict['sim_init_year']
comparison_year = param_dict['comparison_year']
subsector_to_calibrate = param_dict['subsector_to_calibrate']
error_type = param_dict['error_type']
weight_type = param_dict['weight_type']
unique_id = datetime.now().strftime("%Y%m%d%H%M%S")
swarm_size = param_dict['swarmsize']
maxiter = param_dict['maxiter']
# input_rows = param_dict['input_rows']
ssp_edgar_cw_file_name = param_dict['ssp_edgar_cw']

logging.info(f"Starting optimization for {target_region} (ISO code: {iso_alpha_3})")
# logging.info(f"Input rows: {input_rows}")
logging.info(f"Stressed variables report version: {stressed_variables_report_version}")
logging.info(f"Input data file to calibrate: {input_data_file_to_calibrate}")
logging.info(f"Energy model flag: {energy_model_flag}")
logging.info(f"Use EDGAR DB flag: {use_edgar_db_flag}")
logging.info(f"Simulation initial year: {sim_init_year}")
logging.info(f"Comparison year: {comparison_year}")
logging.info(f"Subsector to calibrate: {subsector_to_calibrate}")
logging.info(f"Error type: {error_type}")
logging.info(f"Weight type: {weight_type}")
logging.info(f"Unique ID: {unique_id}")
logging.info(f"Swarm size: {swarm_size}")
logging.info(f"Max iterations: {maxiter}")
logging.info(f"Detailed diff report flag: {detailed_diff_report_flag}")
logging.info(f"SSP-EDGAR crosswalk file: {ssp_edgar_cw_file_name}")

2025-07-22 17:07:38,157 - INFO - Starting optimization for uganda (ISO code: UGA)
2025-07-22 17:07:38,158 - INFO - Stressed variables report version: stressed_variables_report_2025_06_06.xlsx
2025-07-22 17:07:38,158 - INFO - Input data file to calibrate: new_uganda_inputs.csv
2025-07-22 17:07:38,159 - INFO - Energy model flag: False
2025-07-22 17:07:38,160 - INFO - Use EDGAR DB flag: False
2025-07-22 17:07:38,161 - INFO - Simulation initial year: 2015
2025-07-22 17:07:38,161 - INFO - Comparison year: 2022
2025-07-22 17:07:38,162 - INFO - Subsector to calibrate: None
2025-07-22 17:07:38,163 - INFO - Error type: mse
2025-07-22 17:07:38,164 - INFO - Weight type: norm_weight
2025-07-22 17:07:38,164 - INFO - Unique ID: 20250722170738
2025-07-22 17:07:38,165 - INFO - Swarm size: 346
2025-07-22 17:07:38,165 - INFO - Max iterations: 10
2025-07-22 17:07:38,166 - INFO - Detailed diff report flag: True
2025-07-22 17:07:38,167 - INFO - SSP-EDGAR crosswalk file: sisepuede_edgar_active_crosswalk.csv

In [7]:
# Make sure the output directory exists
os.makedirs(OPT_OUTPUT_PATH, exist_ok=True)

# Make sure pso output directories exist
PSO_OUTPUT_PATH = build_path([OUTPUT_PATH, target_region])
os.makedirs(PSO_OUTPUT_PATH, exist_ok=True)

# Create the output directory for the PSO results using the unique ID
RUN_OUTPUT_DIR = os.path.join(PSO_OUTPUT_PATH, unique_id)
os.makedirs(RUN_OUTPUT_DIR, exist_ok=True)

In [8]:
# Save the config file to the output directory
config_file_name = f"{unique_id}_config.yaml"
config_file_output_path = os.path.join(RUN_OUTPUT_DIR, config_file_name)
helper_functions.copy_param_yaml(build_path([OPT_CONFIG_FILES_PATH, yaml_file]), config_file_output_path)


In [9]:
# Load input dataset
REAL_DATA_FILE_PATH = build_path([DATA_PATH, input_data_file_to_calibrate])
examples = SISEPUEDEExamples()
cr = examples("input_data_frame")
df_input = pd.read_csv(REAL_DATA_FILE_PATH)

# Add missing columns and reformat the input datas
df_input = df_input.rename(columns={'period': 'time_period'})
df_input = gu.add_missing_cols(cr, df_input.copy())
df_input = df_input.drop(columns='iso_code3', errors='ignore')
df_input.head()

,region,time_period,area_gnrl_country_ha,avgload_trns_freight_tonne_per_vehicle_aviation,avgload_trns_freight_tonne_per_vehicle_rail_freight,avgload_trns_freight_tonne_per_vehicle_road_heavy_freight,avgload_trns_freight_tonne_per_vehicle_water_borne,avgmass_lvst_animal_buffalo_kg,avgmass_lvst_animal_cattle_dairy_kg,avgmass_lvst_animal_cattle_nondairy_kg,...,ef_lndu_conv_shrublands_to_grasslands_gg_co2_ha,ef_lndu_conv_shrublands_to_other_gg_co2_ha,ef_lndu_conv_shrublands_to_pastures_gg_co2_ha,ef_lndu_conv_shrublands_to_settlements_gg_co2_ha,ef_lndu_conv_shrublands_to_shrublands_gg_co2_ha,ef_lndu_conv_shrublands_to_wetlands_gg_co2_ha,ef_lndu_conv_wetlands_to_flooded_gg_co2_ha,ef_lndu_conv_wetlands_to_pastures_gg_co2_ha,ef_lndu_conv_wetlands_to_shrublands_gg_co2_ha,gasrf_lsmm_biogas_anaerobic_lagoon
0,uganda,0,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,0.049159,0.052189,0.049159,0.052189,0.0,0.049159,0.003030,0.0,0.0,0.0
1,uganda,1,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,0.049179,0.052189,0.049179,0.052189,0.0,0.049179,0.003010,0.0,0.0,0.0
2,uganda,2,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,0.049229,0.052189,0.049229,0.052189,0.0,0.049229,0.002960,0.0,0.0,0.0
3,uganda,3,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,0.049239,0.052189,0.049239,0.052189,0.0,0.049239,0.002949,0.0,0.0,0.0
4,uganda,4,24155000.0,70.0,2923.0,31.751466,6468.0,315.0,508.0,303.0,...,0.049239,0.052189,0.049239,0.052189,0.0,0.049239,0.002949,0.0,0.0,0.0


In [10]:
# Load the stressed variables mapping file
stressed_vars_mapping = pd.read_excel(build_path([VAR_MAPPING_FILES_PATH, stressed_variables_report_version]))
stressed_vars_mapping

,variable_name,subsector,is_stressed,l_bound,u_bound,group_id,needs_check,is_capped,is_asymmetric
0,ratio_ippu_production_to_harvested_wood_demand...,IPPU,1,0.5,1.5,246.0,NaN,0,0
1,ratio_ippu_production_to_harvested_wood_demand...,IPPU,1,0.5,1.5,246.0,NaN,0,0
2,pop_lvst_initial_sheep,AFOLU,1,0.1,10.0,245.0,NaN,0,1
3,pop_lvst_initial_pigs,AFOLU,1,0.1,1.9,244.0,NaN,0,0
4,pop_lvst_initial_mules,AFOLU,1,0.1,1.9,243.0,NaN,0,0
...,...,...,...,...,...,...,...,...,...
2211,yf_agrc_pulses_tonne_ha,AFOLU,0,0.5,1.5,NaN,NaN,0,0
2212,yf_agrc_rice_tonne_ha,AFOLU,0,0.5,1.5,NaN,NaN,0,0
2213,yf_agrc_sugar_cane_tonne_ha,AFOLU,0,0.5,1.5,NaN,NaN,0,0
2214,yf_agrc_tubers_tonne_ha,AFOLU,0,0.5,1.5,NaN,NaN,0,0


In [11]:
# Subset the stressed variables mapping file to is_stressed = 1
stressed_vars_mapping = stressed_vars_mapping[stressed_vars_mapping['is_stressed'] == 1]

# Check for nulls in the is_stressed column
if stressed_vars_mapping['is_stressed'].isnull().sum() > 0:
    raise ValueError("There are null values in the is_stressed column of the stressed variables mapping file.")


In [12]:
# Reset the index of the stressed variables mapping file
stressed_vars_mapping = stressed_vars_mapping.reset_index(drop=True)

# Set group_id as integer
stressed_vars_mapping['group_id'] = stressed_vars_mapping['group_id'].astype(int)


In [14]:
# Check group id array
stressed_vars_mapping.group_id.unique()

array([246, 245, 244, 243, 242, 241, 240, 239, 238, 237, 236, 235, 234,
       233, 232, 231, 230, 229, 228, 227, 226, 225, 224, 223, 222, 221,
       220, 219, 218, 217, 216, 215, 214, 213, 212, 211, 210, 209, 208,
       207, 206, 205, 204, 203, 202, 201, 200, 199, 198, 197, 196, 195,
       194, 193, 192, 191, 190, 189, 188, 187, 186, 185, 184, 183, 182,
       181, 180, 179, 178, 177, 176, 175, 174, 173, 172, 171, 170, 169,
       168, 167, 166, 165, 164, 163, 162, 161, 160, 159, 158, 157, 156,
       155, 154, 153, 152, 151, 150, 149, 148, 147, 146, 145, 144, 143,
       142, 141, 140, 139, 138, 137, 136, 135, 134, 133, 132, 131, 130,
       129, 128, 127, 126, 125, 124, 123, 122, 121, 120, 119, 118, 117,
       116, 115, 114, 113, 112, 111, 110, 109, 108, 107, 106, 105, 104,
       103, 102, 101, 100,  99,  98,  97,  96,  95,  94,  93,  92,  91,
        90,  89,  88,  87,  86,  85,  84,  83,  82,  81,  80,  79,  78,
        77,  76,  75,  74,  73,  72,  71,  70,  69,  68,  67,  6

In [ ]:
# Make sure stressed_vars_mapping is sorted by group_id
stressed_vars_mapping = stressed_vars_mapping.sort_values(by='group_id', ascending=True)

# Get group ids of the vars that are stressed
group_ids = stressed_vars_mapping[stressed_vars_mapping["is_stressed"] == 1]["group_id"].unique()
n_groups = len(group_ids)

logging.info(f"Number of groups to stress: {n_groups}")

2025-07-22 17:09:46,180 - INFO - Number of groups to stress: 247


In [16]:
# Create a dictionary with the group ids as keys and the corresponding variable names as values
group_vars_dict = {}
for group_id in group_ids:
    group_vars_dict[group_id] = stressed_vars_mapping[stressed_vars_mapping["group_id"] == group_id]["variable_name"].values


In [17]:
# Crear un nuevo diccionario con claves reenumeradas del 1 a n
reordered_dict = {new_id: group_vars_dict[old_id] for new_id, old_id in enumerate(group_vars_dict, 0)}
# logging.info(f"Group variables dictionary: {reordered_dict}")

In [18]:
# Get the lower and upper bounds for each group
l_bounds = stressed_vars_mapping.groupby("group_id")["l_bound"].first().values
u_bounds = stressed_vars_mapping.groupby("group_id")["u_bound"].first().values

print(f"{10*'#'}bounds arrays{10*'#'}")
print(l_bounds)
print(u_bounds)

print(f"{10*'#'}bounds vector size{10*'#'}")
print('l_bounds:', len(l_bounds))
print('u_bounds:', len(u_bounds))

##########bounds arrays##########
[0.5  0.6  0.8  0.8  0.8  0.8  0.8  0.8  0.8  0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.1  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.8  0.8  0.8  0.8  0.8  0.8
 0.8  0.8  0.8  0.8  0.8  0.8  0.8  0.8  0.8  0.8  0.5  0.75 0.8  0.7
 0.75 0.75 0.8  0.75 0.75 0.75 0.75 0.8  0.75 0.8  0.8  0.95 0.75 0.75
 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75
 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75
 0.75 0.75 0.75 0.75 0.75 0.75 0.5  0.6  0.6  0.6  0.6  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.75 0.75 0.75 0.75 0.75 0.95 0.75 0.75 0.7  0.75
 0.75 0.5  0.75 0.75 0.75 0.75 0.75 0.75 0.75 0.75 0

In [20]:
# Initialize the ErrorFunctions class
ef = ErrorFunctions()

#  Initialize the DiffReportUtils class
edgar_ssp_cw_path = build_path([SECTORAL_REPORT_MAPPING_PATH, ssp_edgar_cw_file_name])
dru = DiffReportUtils(iso_alpha_3, edgar_ssp_cw_path, SECTORAL_REPORT_PATH, energy_model_flag, sim_init_year=sim_init_year, comparison_year=comparison_year)

In [21]:
edgar_emission_db_path = build_path([SECTORAL_REPORT_MAPPING_PATH, 'emission_targets_uganda.csv'])
edgar_df = dru.get_edgar_region_df(edgar_emission_db_path)
edgar_df.head()

,ids,iso_alpha_3,edgar_emission,year
0,1:lvst-lsmm:ch4,UGA,9.972244,2022
1,1:lvst-lsmm:ch4,UGA,9.972244,2022
2,3:lsmm:n2o,UGA,0.299059,2022
3,4:agrc:co2,UGA,0.000000,2022
4,5:agrc:ch4,UGA,0.576887,2022


In [22]:
# Check if the EDGAR DataFrame is empty
logging.info(f"EDGAR DataFrame shape: {edgar_df.shape}")
if edgar_df.empty:
    raise ValueError(f"EDGAR DataFrame is empty. Please check that your iso_alpha_3 is in {edgar_emission_db_path}")

# Initialize global variable to store the previous calculated error
previous_error = float('inf')

# Initialize global variable to store the worst_valid_error
worst_valid_error = float(12)

# Initialize the SSP model
ssp_model = SSPModelForCalibration(energy_model_flag=energy_model_flag)


2025-07-22 17:12:13,653 - INFO - EDGAR DataFrame shape: (63, 4)
2025-07-22 17:12:14,458 - INFO - Successfully initialized SISEPUEDEFileStructure.
2025-07-22 17:12:14,458 - INFO - Successfully initialized SISEPUEDEFileStructure.
2025-07-22 17:12:14,462 - WARNING - Missing key dict_dimensional_keys: key time_series not found. Tables that rely on the time_series will not have index checking.
2025-07-22 17:12:14,462 - WARNING - Missing key dict_dimensional_keys: key time_series not found. Tables that rely on the time_series will not have index checking.
2025-07-22 17:12:14,463 - INFO - 	Setting export engine to 'csv'.
2025-07-22 17:12:14,463 - INFO - 	Setting export engine to 'csv'.
2025-07-22 17:12:14,464 - WARNING - No index fields defined. Index field values will not be checked when writing to tables.
2025-07-22 17:12:14,464 - WARNING - No index fields defined. Index field values will not be checked when writing to tables.
2025-07-22 17:12:14,465 - INFO - Successfully instantiated table

yay


2025-07-22 17:12:15,635 - INFO - Initializing FutureTrajectories
2025-07-22 17:12:15,635 - INFO - Initializing FutureTrajectories
2025-07-22 17:12:20,701 - INFO - Instantiating 1392 sampling units.
2025-07-22 17:12:20,701 - INFO - Instantiating 1392 sampling units.
2025-07-22 17:12:20,750 - INFO - Iteration 0 complete.
2025-07-22 17:12:20,750 - INFO - Iteration 0 complete.
2025-07-22 17:12:32,426 - INFO - Iteration 250 complete.
2025-07-22 17:12:32,426 - INFO - Iteration 250 complete.
2025-07-22 17:12:37,291 - INFO - Iteration 500 complete.
2025-07-22 17:12:37,291 - INFO - Iteration 500 complete.
2025-07-22 17:12:42,568 - INFO - Iteration 750 complete.
2025-07-22 17:12:42,568 - INFO - Iteration 750 complete.
2025-07-22 17:12:47,509 - INFO - Iteration 1000 complete.
2025-07-22 17:12:47,509 - INFO - Iteration 1000 complete.
2025-07-22 17:12:52,593 - INFO - Iteration 1250 complete.
2025-07-22 17:12:52,593 - INFO - Iteration 1250 complete.
2025-07-22 17:12:55,402 - INFO - 	1392 sampling un

In [25]:
# Simulation model
def simulation_model(df: pd.DataFrame) -> pd.DataFrame:
    """
    Function that simulates outputs based on the scaled inputs.
    """
    sim_output_df = ssp_model.run_ssp_simulation(df)
    
    # Handle empty DataFrame
    if sim_output_df is None or sim_output_df.empty:
        logging.warning("Simulation Output DataFrame is empty. Returning an empty DataFrame.")
        return pd.DataFrame()

    return sim_output_df


In [26]:
# Define the objective function
def objective_function(x):
    
    # Global variables
    global worst_valid_error
    global previous_error
    global edgar_df
    
    
    # x: scaling factors for each group_id
    modified_df = df_input.copy()
    
    # TODO: Vectorize this loop
    # Scale the variables per group
    for group_id in group_vars_dict:
        for var in group_vars_dict[group_id]:
            modified_df[var] = modified_df[var] * x[group_id]
    
    
    if normalization_flag:
        # Handle frac var group normalization
        processed_input_df = helper_functions.simple_frac_normalization(modified_df, frac_vars_mapping)

        # Clip the variables
        processed_input_df = helper_functions.clip_values(processed_input_df, vars_to_clip)
    
    else:
        processed_input_df = modified_df.copy()

    
    # Run the model
    sim_output_df = simulation_model(processed_input_df)
    
    # Assing a penalty if the simulation output is empty
    if sim_output_df.empty:
        error_val = worst_valid_error * 1.1  # Slighly higher than the worst valid error for invalid outputs
        logging.warning("Simulation returned an empty DataFrame. Setting Error to a penalty value.")
    
    else:
        # Generate diff reports to calculate Error
        dru.run_report_generator(edgar_emission_df=edgar_df, ssp_out_df=sim_output_df)

        
        # Checks if the model failed in any subsector
        if dru.model_failed_flag:
            error_val = worst_valid_error * 1.1  # Slighly higher than the worst valid error for invalid outputs
            logging.warning("Model failed in a subsector. Setting Error to a penalty.")
        
        # Calculate error
        elif detailed_diff_report_flag:
            error_val = ef.calculate_error(error_type, dru.sectoral_emission_report, weight_type)
        else:
            error_val = ef.calculate_error(error_type, dru.subsector_emission_report, weight_type)

    # Update worst_valid_error
    if error_val > worst_valid_error:
        worst_valid_error = error_val
        logging.info(f"New worst_valid_error: {worst_valid_error:.6f}")

    # Log the error
    logging.info("=" * 30)
    logging.info(f"Current ERROR: {error_val:.6f}")
    logging.info("=" * 30)

    # Log the scaling factors and the error
    helper_functions.log_to_csv(x, error_val, error_type, RUN_OUTPUT_DIR, target_region, unique_id)

    # Save the processed_input_df, detailed_diff_report and subsector_diff_report if the error is less than the previous error
    if error_val < previous_error:
        previous_error = error_val
        processed_input_df.to_csv(build_path([RUN_OUTPUT_DIR, f"best_input_df_{unique_id}.csv"]), index=False)
        dru.sectoral_emission_report.to_csv(build_path([RUN_OUTPUT_DIR, f"best_detailed_diff_report_{unique_id}.csv"]), index=False)
        dru.subsector_emission_report.to_csv(build_path([RUN_OUTPUT_DIR, f"best_subsector_diff_report_{unique_id}.csv"]), index=False)
        logging.info(f"Best Input Data and Diff Reports Updated to {RUN_OUTPUT_DIR}")

    return error_val

In [27]:
# Initialize the PSO optimizer
best_solution, best_value = pso(
    objective_function,
    l_bounds,
    u_bounds,
    swarmsize=swarm_size,
    maxiter=maxiter,
    )

2025-02-19 16:31:10,848 - INFO - Running AFOLU model
2025-02-19 16:31:10,848 - INFO - Running AFOLU model
2025-02-19 16:31:10,992 - INFO - AFOLU model run successfully completed
2025-02-19 16:31:10,992 - INFO - AFOLU model run successfully completed
2025-02-19 16:31:10,993 - INFO - Running CircularEconomy model
2025-02-19 16:31:10,993 - INFO - Running CircularEconomy model
2025-02-19 16:31:11,042 - INFO - CircularEconomy model run successfully completed
2025-02-19 16:31:11,042 - INFO - CircularEconomy model run successfully completed
2025-02-19 16:31:11,043 - INFO - Running IPPU model
2025-02-19 16:31:11,043 - INFO - Running IPPU model
2025-02-19 16:31:11,120 - INFO - IPPU model run successfully completed
2025-02-19 16:31:11,120 - INFO - IPPU model run successfully completed
2025-02-19 16:31:11,121 - INFO - Running Energy model (EnergyConsumption without Fugitive Emissions)
2025-02-19 16:31:11,121 - INFO - Running Energy model (EnergyConsumption without Fugitive Emissions)
2025-02-19 1

Stopping search: maximum iterations reached --> 2


In [28]:
# logging.info(f"Best scaling vector: {best_solution}")
logging.info(f"Best error: {best_value}")

2025-02-19 16:31:16,114 - INFO - Best error: 7.063751399167127
